# B2-020 — Session 1: Train Token Embeddings

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).



## 1. One-hot lookup and shapes

For vocabulary size $V=12$, a token ID becomes a one-hot row in $\mathbb R^{12}$. Multiplying `(B,V) @ (V,D)` with `D=8` selects rows of the trainable embedding table exactly; `nn.Embedding` is the indexed form of the same operation.

**Checkpoint 1A.** Trace `(3,12) @ (12,8)`.

**Checkpoint 1B.** Which table rows are selected by IDs `[4,4,7]`?

In [ ]:
import torch
torch.manual_seed(20260812)
ATOL = 1e-6
RTOL = 1e-6
assert torch.get_num_threads() >= 1

## 2. A learned table is a model parameter

Fixed vectors are supplied values. A learned-token-embedding table begins at a seeded random state, has `requires_grad=True`, and changes only because a predictive loss sends gradients into selected rows. This is embedding-model-training, not a vocabulary lookup trick.

**Checkpoint 2A.** State the fixed-versus-learned boundary.

**Checkpoint 2B.** Why does a seed not make a table pretrained?

## 3. Context-to-target objective

Use one context token to predict a target token. The selected embedding row feeds a distinct linear vocabulary head, producing logits of shape `(B,12)`. Mean cross-entropy compares those logits with integer targets of shape `(B,)`.

**Checkpoint 3A.** Name the logit and target dtypes.

**Checkpoint 3B.** Why must the target not be part of the input context?

## 4. Worked example: one gradient update

Start with a displayed `(12,8)` table, select row 4 with a one-hot vector, compute 12 logits, apply stable softmax, take `-log p[target]`, call `zero_grad(set_to_none=True)`, `backward()`, and one optimizer `step()`. Only the selected lookup row receives a direct embedding gradient; the vocabulary head also changes.

**Checkpoint 4A.** Write the scalar loss for target ID 6.

**Checkpoint 4B.** Audit which parameter groups move.

## 5. Seeded embedding training

Seed Python, NumPy, and Torch with `20260812`. Train the literal tiny context pairs in stored order. Record initial and final loss rather than asserting that every random-looking table is learned.

**Checkpoint 5A.** What makes two runs reproducible?

**Checkpoint 5B.** Why is a final loss alone weak evidence?

## 6. Nearest-neighbor interpretation

Cosine similarity is an analysis of the learned rows, not the training objective here. Compare neighbors before and after training and connect any change to shared predictive contexts; do not import an external corpus or GloVe file.

**Checkpoint 6A.** What does a changed neighbor certify?

**Checkpoint 6B.** What does it not certify?

## 7. Common pitfalls

Broken patterns include detaching the selected row, passing probabilities rather than logits to cross-entropy, training a fresh table but reporting an old copy, and claiming all rows receive direct lookup gradient. Repair each by tracing parameter identity, shape, and gradient flow.

**Checkpoint 7A.** Which bug produces no row movement?

**Checkpoint 7B.** Which bug double-applies softmax?

## 8. Exam connections and Going deeper

An exam can ask for row-gradient support, exact one-hot multiplication, normalized target probability, or a loss-change certificate. Going deeper is forward-only: negative sampling and weight tying are named but are not used in this unit's required protocol.

**Checkpoint 8A.** Which practices revisit row support?

**Checkpoint 8B.** Why is weight tying outside the pinned checkpoint?